In [51]:
!uv pip install numpy pandas matplotlib nltk pythainlp

Checked 5 packages in 15ms


In [52]:
import pandas as pd
import numpy as np
import nltk

In [53]:
df = pd.read_csv('./financial-news-data/financial-news-train.csv', encoding='utf-8')

In [54]:
# Show full text of the first row
pd.set_option('display.max_colwidth', None)

# De-capitalize
df['text'] = df.text.str.lower()

# Remove quotes
df['text'] = df.text.str.replace(r'"', "", regex=True)

# Remove links
match_link = r'https:\/\/(.)+(\s|$)'
df['text'] = df.text.str.replace(match_link, "", regex=True)

# Remove any word that doesn't start with alphabetic character
match_non_word = r'[^(a-zA-Z)\s].*?(\s|$)'
df['text'] = df.text.str.replace(match_non_word, "", regex=True)

# Remove non-alphabetic characters
match_non_alpha = r'[^a-zA-Z]\s'
df['text'] = df.text.str.replace(match_non_alpha, "", regex=True)

df.drop_duplicates(subset=['text'], inplace=True)
df

,text,label
0,here are thursdaybiggest analyst callsappleamazonteslapalantirdocusignexxon more,Analyst Update
1,buy las vegas sands as travel to singapore buildswells fargo says,Analyst Update
2,piper sandler downgrades docusign to sellciting elevated risks amid ceo transition,Analyst Update
3,analysts react to teslalatest earningsbreak down whatnext for electric car maker,Analyst Update
4,netflix and its peers are set for a to growthanalysts saygiving one stock upside,Analyst Update
...,...,...
16984,china developers face wall of dollar bond payments in second half,Treasuries | Corporate Debt
16985,kfw credit line for uniper could be raised to bln eur handelsblatt,Treasuries | Corporate Debt
16987,russian,Treasuries | Corporate Debt
16988,global esg bond issuance posts hdip as supranationals cut back,Treasuries | Corporate Debt


In [55]:
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /home/kami/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/kami/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/kami/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [56]:
# Show full text
pd.set_option('display.max_colwidth', None)

# Lowercase
df['text'] = df['text'].str.lower()

# Remove quotes
df['text'] = df['text'].str.replace('"', '', regex=False)

# Remove links
df['text'] = df['text'].str.replace(r'https?://\S+', '', regex=True)

# Remove words that don't start with an alphabetic character
df['text'] = df['text'].str.replace(r'\b[^a-z\s]\S*', '', regex=True)

# Remove non-alphabetic characters
df['text'] = df['text'].str.replace(r'[^a-z\s]', '', regex=True)

# Normalize whitespace after removals
df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()

df.drop_duplicates(subset=['text'], inplace=True)

# NLTK remove stopwods
stop_words = set(stopwords.words('english'))
df['text'] = df['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

df

,text,label
0,thursdaybiggest analyst callsappleamazonteslapalantirdocusignexxon,Analyst Update
1,buy las vegas sands travel singapore buildswells fargo says,Analyst Update
2,piper sandler downgrades docusign sellciting elevated risks amid ceo transition,Analyst Update
3,analysts react teslalatest earningsbreak whatnext electric car maker,Analyst Update
4,netflix peers set growthanalysts saygiving one stock upside,Analyst Update
...,...,...
16984,china developers face wall dollar bond payments second half,Treasuries | Corporate Debt
16985,kfw credit line uniper could raised bln eur handelsblatt,Treasuries | Corporate Debt
16987,russian,Treasuries | Corporate Debt
16988,global esg bond issuance posts hdip supranationals cut back,Treasuries | Corporate Debt


In [57]:
vocabs = set()

for text in df.text:
    vocabs.update(text.split())

print(len(vocabs))
print(vocabs)

30242
{'prohibiting', 'allergy', 'muchanalyst', 'companiesreliable', 'tg', 'unambiguous', 'ratesand', 'zimvie', 'shortfall', 'urecalls', 'prebath', 'indiawholesale', 'sheikh', 'bankinterqlending', 'hikessmaller', 'nikkei', 'saysaddingfor', 'bom', 'ueye', 'hasbroincqresults', 'gibsonjoins', 'stunt', 'forward', 'shoigu', 'thielcrypto', 'heleaving', 'americafrom', 'commentscfrarecap', 'investments', 'tapped', 'threadand', 'watchamerican', 'delfonics', 'vwap', 'analyze', 'cleo', 'savingsone', 'midto', 'freedom', 'insteadso', 'newsitalian', 'capitalbiggest', 'opportunityfundamentals', 'insane', 'eni', 'weekpower', 'forms', 'intelligencedatabase', 'marchaccording', 'tessco', 'companystock', 'offsmallstrategy', 'theredelta', 'secondor', 'antibless', 'permits', 'usnr', 'qestimateswhile', 'lands', 'cachexia', 'vans', 'sharein', 'catalyst', 'accelerant', 'parmeshwaran', 'roadblock', 'healthprofessiona', 'daypaused', 'biorecycling', 'seng', 'gainsstrong', 'molninvestors', 'scholastic', 'bikes', '

In [58]:
# Create empty feature vectors
features = pd.DataFrame(np.zeros((len(df), len(vocabs)), dtype=int), columns=sorted(vocabs), index=df.index)

In [62]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(vocabulary=sorted(vocabs))
X = vectorizer.fit_transform(df['text'])

In [64]:
vectorizer.get_feature_names_out()

array(['aa', 'aabeats', 'aad', ..., 'zynx', 'zyus', 'zyversa'],
      shape=(30242,), dtype=object)

In [67]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(df['label'])

In [68]:
encoder.classes_

array(['Analyst Update', 'Company | Product News', 'Currencies',
       'Dividend', 'Earnings', 'Energy | Oil', 'Fed | Central Banks',
       'Financials', 'General News | Opinion',
       'Gold | Metals | Materials', 'IPO', 'Legal | Regulation',
       'M&A | Investments', 'Macro', 'Markets', 'Personnel Change',
       'Politics', 'Stock Commentary', 'Stock Movement',
       'Treasuries | Corporate Debt'], dtype=object)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X, y)